In [ ]:
!git clone https://github.com/pohaoc2/share_space.git
project_dir = '/content/share_space'
%cd {project_dir}
!git checkout chiu/CBM
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

In [ ]:
%cd /content/share_space/src/share_space/
!mkdir checkpoints
%cd checkpoints
!mkdir stage_1
%cd stage_1
!pip install -q gdown
!gdown --fuzzy "https://drive.google.com/file/d/1Q4uWslZ-a2KEZxn5FNeEPcaoSW8n6BND/view?usp=sharing" -O best_model.pth

In [ ]:
import os
from IPython.display import clear_output 
sim_exp = True
if sim_exp:
    !pip install -q gdown
    %cd "/content/share_space/"
    !mkdir "data"
    %cd "/content/share_space/data"

    if not os.path.exists("exp.zip"):
        !gdown --fuzzy "https://drive.google.com/file/d/1rAd7nMFZQJce3i_0IJL4dvClRiWeFAm7/view?usp=sharing" -O exp.zip
        !unzip exp.zip
    else:
        print("exp.zip already exists. Skipping download.")

    if not os.path.exists("sim.zip"):
        !gdown --fuzzy "https://drive.google.com/file/d/1aOJuoAfKQjQwc5d4s_nRV6T67pHZqyJG/view?usp=sharing" -O sim.zip
        !unzip sim.zip
    else:
        print("sim.zip already exists. Skipping download.")

else:
    %cd "/content/share_space/"
    !mkdir "data"
    %cd "/content/share_space/data"

    if not os.path.exists("small_style.zip"):
        !wget "https://www.dropbox.com/scl/fi/6vnm43ryn7ewd1log5a5v/small_style.zip?rlkey=ez4zlo0clyg74nmzho09v790i&st=n8v76wl7&dl=0" -O small_style.zip
        !unzip small_style.zip
    else:
        print("small_style.zip already exists. Skipping download.")

    if not os.path.exists("coco.zip"):
        !wget "https://www.dropbox.com/scl/fi/5ws1v8btu12pw8u5038cl/coco.zip?rlkey=ngwuzad2u9sk6t0c47okwgtbc&st=aemohm68&dl=0" -O coco.zip
        #!wget "http://images.cocodataset.org/zips/test2017.zip" -O coco.zip
        !unzip coco.zip

    else:
        print("coco.zip already exists. Skipping download.")
clear_output()

In [ ]:
!pip install pytorch-msssim
clear_output()

In [ ]:
import sys
%env PYTHONPATH=/content/share_space/src
%cd /content/share_space/src/share_space/
!python3 main.py

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
state_names = {
    0: 'Cell Count',
    1: 'OTHER',
    2: 'INFLAMMATORY',
    3: 'HEALTHY_EPITHELIAL',
    4: 'DYSPLASTIC_MALIGNANT',
    5: 'FIBROBLAST',
    6: 'MUSCLE',
    7: 'ENDOTHELIAL'
}
state = 0
image_path = f"visualization/reconstructed_images/reconstructed_images_{state_names[state]}.png"
image = Image.open(image_path)
from IPython.display import display
display(image)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
state = 0
image_path = f"visualization/style_transfer/style_transfer_{state_names[state]}.png"
image = Image.open(image_path)
from IPython.display import display
display(image)

In [ ]:
!pip install -U diffusers transformers accelerate
clear_output()

In [ ]:
import yaml
%env PYTHONPATH=/content/share_space/src
%cd /content/share_space/src/share_space/
import dataset
import importlib
importlib.reload(dataset)

from dataset import get_real_dataloaders
def load_config(config_path="config.yaml"):
    """Load configuration from YAML file."""
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
    return config
config_path = "config.yaml"
config = load_config(config_path)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Create dataloaders
print("Creating dataloaders...")
train_loader, val_loader = get_real_dataloaders(
    exp_dir=config["data"]["exp_dir"],
    sim_dir=config["data"]["sim_dir"],
    batch_size=config["data"]["batch_size"],
    num_workers=config["data"]["num_workers"],
    img_size=config["model"]["img_size"],
    train_split=config["data"]["train_split"],
    seed=config["data"]["seed"],
)

In [ ]:
import torch
from diffusers import DiffusionPipeline
from diffusers import AutoencoderKL

device = torch.device('cuda')

import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# We do not host the weights of the SD3 VAE -- load it from StabilityAI
sd3_vae = AutoencoderKL.from_pretrained("stabilityai/stable-diffusion-3.5-large", subfolder="vae")

pipeline = DiffusionPipeline.from_pretrained(
    "StonyBrook-CVLab/PixCell-256-Cell-ControlNet",
    vae=sd3_vae,
    custom_pipeline="pohaoc2/PixCell-pipeline-ControlNet-fork",
    trust_remote_code=True,
)

pipeline.to(device);


In [ ]:
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

timm_kwargs = {
            'img_size': 224,
            'patch_size': 14,
            'depth': 24,
            'num_heads': 24,
            'init_values': 1e-5,
            'embed_dim': 1536,
            'mlp_ratio': 2.66667*2,
            'num_classes': 0,
            'no_embed_class': True,
            'mlp_layer': timm.layers.SwiGLUPacked,
            'act_layer': torch.nn.SiLU,
            'reg_tokens': 8,
            'dynamic_img_size': True
        }
uni_model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)
uni_transforms = create_transform(**resolve_data_config(uni_model.pretrained_cfg, model=uni_model))
uni_model.eval()
uni_model.to(device);


In [ ]:
import numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download
from utils import inverse_transform
import matplotlib.pyplot as plt
%cd /content/share_space/src/share_space/
os.makedirs("generated_images", exist_ok=True)
os.makedirs("masks", exist_ok=True)
os.makedirs("exp_images", exist_ok=True)
guidance_scale = 1
for b_id, batch in enumerate(train_loader):
    generated_exp_like_images = []
    for idx, sim_img in enumerate(batch['simulation']):
        sim_name = batch['sim_binary_path'][idx].split('/')[-1]
        shuffled_name = batch['shuffled_exp_path'][idx].split('/')[-1]
        sim_img = sim_img.permute(1, 2, 0).cpu().numpy()
        exp_img = batch['shuffled_exp'][idx].permute(1, 2, 0).cpu().numpy()

        sim_img = inverse_transform(sim_img)
        exp_img = inverse_transform(exp_img)

        # exp image (PIL, continuous)
        exp_img = Image.fromarray(
            (exp_img * 255).clip(0, 255).astype(np.uint8)
        )
        exp_img = exp_img.resize((256, 256), resample=Image.BILINEAR)

        exp_img.save(f"exp_images/{b_id}_{idx}_{shuffled_name}")
        # sim mask (binary)
        sim_img = Image.fromarray((sim_img*255).astype(np.uint8))
        sim_img = sim_img.resize((256, 256), resample=Image.NEAREST)
        sim_img.save(f"masks/{b_id}_{idx}_{sim_name}")
        sim_img = np.asarray(sim_img)

        # UNI embedding
        uni_inp = uni_transforms(exp_img).unsqueeze(0)
        with torch.inference_mode():
            uni_emb = uni_model(uni_inp.to(device))

        uni_emb = uni_emb.unsqueeze(1)
        uncond = pipeline.get_unconditional_embedding(uni_emb.shape[0])

        samples = pipeline(
            uni_embeds=uni_emb,
            controlnet_input=sim_img.astype(float),
            negative_uni_embeds=uncond,
            guidance_scale=guidance_scale,
            num_images_per_prompt=1
        ).images
        generated_exp_like_images.append(samples[0])
        samples[0].save(f"generated_images/{b_id}_{idx}_generated.png")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(exp_img)
ax[1].imshow(sim_img.astype(float))
ax[2].imshow(samples[0])
for a in ax:
    a.axis('off')
plt.show()

In [ ]:
!mkdir all_outputs_w{guidance_scale}
!mv generated_images masks exp_images all_outputs_w{guidance_scale}

In [ ]:
from google.colab import files
!zip -r all_outputs_w{guidance_scale}.zip all_outputs_w{guidance_scale}
files.download(f"all_outputs_w{guidance_scale}.zip")
clear_output()

In [ ]:
import numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download
from utils import inverse_transform
import matplotlib.pyplot as plt
os.makedirs("generated_images", exist_ok=True)
exp_img = Image.open('benign.png').convert('RGB')

for sim_id in range(1, 10):
    sim_img = Image.open(f"masks/{sim_id}.png").convert('RGB')
    sim_img = sim_img.resize((256, 256), resample=Image.NEAREST)
    sim_img = np.asarray(sim_img)
    
    # UNI embedding
    uni_inp = uni_transforms(exp_img).unsqueeze(0)
    with torch.inference_mode():
        uni_emb = uni_model(uni_inp.to(device))

    uni_emb = uni_emb.unsqueeze(1)
    uncond = pipeline.get_unconditional_embedding(uni_emb.shape[0])

    samples = pipeline(
        uni_embeds=uni_emb,
        controlnet_input=sim_img.astype(float),
        negative_uni_embeds=uncond,
        guidance_scale=3,
        num_images_per_prompt=1
    ).images
    generated_exp_like_images.append(samples[0])
    samples[0].save(f"generated_images/{b_id}_{idx}_{seed}_generated.png")